# Análise dos resultados — melhor configuração de cada algoritmo

Roda **localmente**, sem treinar nada: lê todos os `metadata.json` de
`results/avaliacao_5_particoes/` (rodadas 3–9; a pasta `vazamento/` fica de
fora). Para cada algoritmo, mostra a configuração de maior KS médio no teste
nas 5 partições, com seus hiperparâmetros, protocolo de treino e quantas
configurações daquele algoritmo foram testadas.

Para incluir a rodada 8, traga a branch `results/churn-rodada8-paciencia`
(`git fetch origin results/churn-rodada8-paciencia && git merge FETCH_HEAD`) e rode de novo.

In [ ]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from churn_telecom import relatorio

pd.set_option("display.max_colwidth", 200)
longo = relatorio.carregar_5_particoes(PROJECT_ROOT / "results")
print(f"{longo.groupby(['rodada', 'modelo']).ngroups} configuracoes, {len(longo)} modelos treinados")

## Tabela: melhor de cada algoritmo

In [ ]:
melhores = relatorio.gerar([PROJECT_ROOT / "results"], out_dir=PROJECT_ROOT)
tabela = melhores[["familia", "ks_mean", "ks_std", "auroc_mean", "acuracia_limiar_validacao_mean", "rodada",
                   "modelo", "treino", "codificacao", "transformacao", "iteracoes_particao0",
                   "configuracoes_testadas", "hiperparametros"]].rename(columns={
    "familia": "Algoritmo", "ks_mean": "KS", "ks_std": "KS desvio", "auroc_mean": "AUROC",
    "acuracia_limiar_validacao_mean": "Acurácia (limiar val.)", "iteracoes_particao0": "épocas/árvores (p0)",
    "configuracoes_testadas": "configs testadas"})
tabela.style.format({"KS": "{:.3f}", "KS desvio": "{:.3f}", "AUROC": "{:.3f}",
                     "Acurácia (limiar val.)": "{:.1%}"}, na_rep="-").hide(axis="index")

## Figura: KS do melhor de cada algoritmo

Pontos = média nas 5 partições; barras = ± 1 desvio.

In [ ]:
from IPython.display import Image
Image(PROJECT_ROOT / "reports" / "figures" / "melhores_por_algoritmo_ks.png")

## Quanto do KS é sorte da divisão? (referência para o ranking do semestre anterior)

O ranking do semestre anterior vai de 0,541 a 0,586, e cada valor vem de **uma única divisão
treino/teste aleatória**. Aqui a regressão logística (C=0,1, sem nenhuma busca) é treinada
em 300 divisões 80/20 diferentes. Só a divisão faz o KS de teste variar ~±0,02, e o máximo
passa de 0,61. Por isso o projeto reporta a **média em 5 partições** com teste pareado.
Gerado por `python -m churn_telecom.referencia`, que também registra a regressão logística
nas 5 partições como referência (`logistica_referencia`, C e transformação escolhidos pela validação).

In [ ]:
sorte = pd.read_csv(PROJECT_ROOT / "reports" / "tables" / "sorte_divisao_unica_logistica.csv").ks_teste
display(sorte.describe(percentiles=[.1, .5, .9]).round(4).to_frame().T)
print(f"fracao de divisoes com KS >= 0,586: {(sorte >= 0.5864).mean():.1%} | >= 0,56: {(sorte >= 0.56).mean():.1%}")
Image(PROJECT_ROOT / "reports" / "figures" / "sorte_divisao_unica.png")

## Rodada 8 — efeito da paciência, da busca e das transformações

Só aparece depois de trazer a branch da rodada 8.

In [ ]:
t = PROJECT_ROOT / "reports" / "tables"
if (t / "rodada8_comparacoes_pareadas.csv").exists():
    display(pd.read_csv(t / "rodada8_resumo.csv").head(25))
    display(pd.read_csv(t / "rodada8_comparacoes_pareadas.csv"))
else:
    print("rodada 8 ainda nao disponivel localmente")

## Rodada 9 — busca ampla: o que ajudou

In [ ]:
t = PROJECT_ROOT / "reports" / "tables"
for nome in ("rodada9_comparacoes_pareadas.csv", "rodada9_importancia_hiperparametros.csv"):
    if (t / nome).exists():
        x = pd.read_csv(t / nome)
        display(x if "importancia" not in nome else
                x.sort_values(["algoritmo", "importancia"], ascending=[True, False]).groupby("algoritmo").head(5))
    else:
        print(nome, "ainda nao disponivel localmente")

## Relatório complementar — cada partição como divisão única

Resultados reportados com **uma só divisão** treino/teste não se comparam diretamente
com a média em 5 partições. Aqui cada partição é lida como se fosse a única divisão
(`python -m churn_telecom.particao_unica`; só acrescenta arquivos `particao_unica_*`).

1. **Seleção pela validação** (a leitura honesta): em cada partição, o modelo de maior
   AUROC de validação, como faria quem só tem aquela divisão. Os ensembles e os modelos
   re-treinados em treino+validação ficam de fora.
2. **Melhor de cada algoritmo, partição a partição:** a configuração vencedora pela média
   e o KS dela em cada uma das 5 partições.
3. **Maior KS de teste por algoritmo:** entre todas as configurações e partições. É
   escolhido olhando o teste, então é otimista por construção e serve só de referência.

In [ ]:
t = PROJECT_ROOT / "reports" / "tables"
print("1) selecao pela validacao, por particao")
display(pd.read_csv(t / "particao_unica_selecao_pela_validacao.csv").round(4))
Image(PROJECT_ROOT / "reports" / "figures" / "particao_unica_selecao_pela_validacao.png")

In [ ]:
print("2) melhor configuracao de cada algoritmo, particao a particao")
display(pd.read_csv(t / "particao_unica_melhor_de_cada_algoritmo.csv").round(4))
Image(PROJECT_ROOT / "reports" / "figures" / "particao_unica_melhor_de_cada_algoritmo.png")

In [ ]:
print("3) maior KS de teste por algoritmo (otimista: escolhido olhando o teste)")
pd.read_csv(t / "particao_unica_maior_ks_por_algoritmo.csv").round(4)

### Paciência vs. épocas/árvores fixas (rodada 3 → rodada 8)

Mesmas 5 partições (sementes 0–4), então a diferença é pareada. Atenção: a
rodada 8 também passou a usar a validação sem cópias para a parada, então o
efeito medido é "paciência + protocolo novo".

In [ ]:
r3 = longo[longo.rodada == "rodada3"]
r8 = longo[longo.rodada == "rodada8"]
linhas = []
for alg in ("mlp", "gb", "xgb", "stab", "tabkan", "kan"):
    for antigo, novo in ((f"{alg}_padrao", f"{alg}_padrao_paciencia"), (f"{alg}_optuna", f"{alg}_optuna_paciencia")):
        a = r3[r3.modelo == antigo].set_index("particao").ks
        b = r8[r8.modelo == novo].set_index("particao").ks
        d = (b - a).dropna()
        if len(d) == 5:
            linhas.append({"antes": antigo, "depois": novo, "ks_antes": a.mean(), "ks_depois": b.mean(),
                           "delta": d.mean(), "delta_desvio": d.std()})
pd.DataFrame(linhas) if linhas else print("rodada 8 ainda nao disponivel localmente")